# Inventory Analytics ETL Pipeline

## Objective

This notebook builds a reusable ETL pipeline for inventory analytics.

The pipeline includes:

- Data Profiling
- Schema Validation
- Data Cleaning
- Business Rule Validation
- Feature Engineering

The processed dataset will be used for exploratory data analysis and dashboard development.

In [1]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
sys.path.append(str(project_root))
print(f"Project root: {project_root}")

Project root: C:\Users\r950930\Documents\DS\inventory-analytics


In [2]:
import os
print(os.getcwd())

c:\Users\r950930\Documents\DS\inventory-analytics\notebooks


In [3]:
from src.io import load_csv

## Load data

In [4]:
df=load_csv(r"C:\Users\r950930\Documents\DS\Datawork2026\HD_Supply_Chain_Inventory\supply_chain_dataset1.csv")

In [5]:
display(df.head())
display(df.tail())
display(df.shape)
display(df.columns)

,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
0,2024/1/1,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.95,20.48,0,0,8.52
1,2024/1/2,SKU_1,WH_1,SUP_8,West,17,575,14,379,0,13.95,20.48,0,0,18.63
2,2024/1/3,SKU_1,WH_1,SUP_8,North,35,540,14,379,0,13.95,20.48,1,0,39.62
3,2024/1/4,SKU_1,WH_1,SUP_8,South,24,516,14,379,0,13.95,20.48,0,0,19.43
4,2024/1/5,SKU_1,WH_1,SUP_8,West,21,495,14,379,0,13.95,20.48,0,0,18.70


,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
91245,2024/12/26,SKU_50,WH_5,SUP_10,South,17,352,7,283,0,6.65,9.6,0,0,19.82
91246,2024/12/27,SKU_50,WH_5,SUP_10,South,21,331,7,283,0,6.65,9.6,0,0,27.96
91247,2024/12/28,SKU_50,WH_5,SUP_10,East,17,314,7,283,0,6.65,9.6,0,0,22.13
91248,2024/12/29,SKU_50,WH_5,SUP_10,South,24,290,7,283,0,6.65,9.6,1,0,24.11
91249,2024/12/30,SKU_50,WH_5,SUP_10,West,15,275,7,283,0,6.65,9.6,1,0,20.19


(91250, 15)

Index(['Date', 'SKU_ID', 'Warehouse_ID', 'Supplier_ID', 'Region', 'Units_Sold',
       'Inventory_Level', 'Supplier_Lead_Time_Days', 'Reorder_Point',
       'Order_Quantity', 'Unit_Cost', 'Unit_Price', 'Promotion_Flag',
       'Stockout_Flag', 'Demand_Forecast'],
      dtype='str')

In [6]:
df.isnull().sum()

Date                       0
SKU_ID                     0
Warehouse_ID               0
Supplier_ID                0
Region                     0
Units_Sold                 0
Inventory_Level            0
Supplier_Lead_Time_Days    0
Reorder_Point              0
Order_Quantity             0
Unit_Cost                  0
Unit_Price                 0
Promotion_Flag             0
Stockout_Flag              0
Demand_Forecast            0
dtype: int64

In [7]:
df["Supplier_Lead_Time_Days"].describe()

count    91250.000000
mean         7.984000
std          3.907929
min          2.000000
25%          4.000000
50%          8.000000
75%         11.000000
max         14.000000
Name: Supplier_Lead_Time_Days, dtype: float64

In [8]:
df["Order_Quantity"].value_counts().head(20)

Order_Quantity
0      86223
386       29
426       28
413       28
218       28
440       27
429       27
453       26
257       26
209       26
225       25
201       25
302       25
488       24
301       24
255       24
382       24
478       24
470       23
452       23
Name: count, dtype: int64

## Data profilling

In [9]:
import src.profiling 
print(src.profiling.__file__)

C:\Users\r950930\Documents\DS\inventory-analytics\src\profiling.py


In [10]:
import inspect
import src.profiling

print(inspect.getsource(src.profiling._get_column_summary))

def _get_column_summary(df):
    """
    _get_column_summary generates column summary information of the given DataFrame.
    parameters:
        df (pd.DataFrame): input Dataset.
        return:
            dict: A structured column summary result.
    """
    
    column_summary = {
        "numeric_columns": df.select_dtypes(include='number').columns.tolist(),
        "categorical_columns": df.select_dtypes(include='object').columns.tolist(),
        "datetime_columns": df.select_dtypes(include='datetime').columns.tolist(),
        "unique_counts": df.nunique()

    }
    return column_summary



In [11]:
from src.profiling import generate_profile
import pandas as pd
from pprint import pprint
profile = generate_profile(df)
print("="*60)
print("Basic Information")
print("="*60)
pprint(display(profile["basic_info"]))

units_sold
Q1 = 13.0
Q3 = 27.0
IQR = 14.0
Lower = -8.0
Upper = 48.0
inventory_level
Q1 = 370.0
Q3 = 564.0
IQR = 194.0
Lower = 79.0
Upper = 855.0
reorder_point
Q1 = 252.0
Q3 = 346.0
IQR = 94.0
Lower = 111.0
Upper = 487.0
order_quantity
Q1 = 0.0
Q3 = 0.0
IQR = 0.0
Lower = 0.0
Upper = 0.0
unit_cost
Q1 = 8.18
Q3 = 16.32
IQR = 8.14
Lower = -4.030000000000001
Upper = 28.53
unit_price
Q1 = 12.0
Q3 = 23.39
IQR = 11.39
Lower = -5.085000000000001
Upper = 40.475
Basic Information


{'sample_data':        date sku_id warehouse_id supplier_id region  units_sold  \
 0  2024/1/1  SKU_1         WH_1       SUP_8   West          10   
 1  2024/1/2  SKU_1         WH_1       SUP_8   West          17   
 2  2024/1/3  SKU_1         WH_1       SUP_8  North          35   
 3  2024/1/4  SKU_1         WH_1       SUP_8  South          24   
 4  2024/1/5  SKU_1         WH_1       SUP_8   West          21   
 
    inventory_level  supplier_lead_time_days  reorder_point  order_quantity  \
 0              592                       14            379               0   
 1              575                       14            379               0   
 2              540                       14            379               0   
 3              516                       14            379               0   
 4              495                       14            379               0   
 
    unit_cost  unit_price  promotion_flag  stockout_flag  demand_forecast  
 0      13.95       20.48    

None


In [12]:
print("="*60)
print("Data Quality")
print("="*60)
display(pd.DataFrame(profile["data_quality"]["missing_values"]).T)
display(pd.DataFrame(profile["data_quality"]["missing_rate"]).T)
display(profile["data_quality"]["duplicate_rows"])
outlier_summary = []

for col, info in profile["data_quality"]["outliers"].items():

    outlier_summary.append({
        "Column": col,
        "Method": info["method"],
        "Count": info["count"],
        "Rate": f"{info['rate']:.2%}",
        "Status": info["status"]
    })

display(pd.DataFrame(outlier_summary))
     
print("="*60)
print("Schema Validation")
print("="*60)
display(pd.DataFrame(profile["schema_validation"]).T)
print("="*60)
print("Column Summary")
print("="*60)
# display(pd.DataFrame(profile["column_summary"]["numeric_columns"]).T) 
display(profile["column_summary"])
# display(profile["column_summary"]["unique_values"])
print("="*60)
print("Descriptive Statistics")
print("="*60)
display(profile["descriptive_statistics"])

Data Quality


,date,sku_id,warehouse_id,supplier_id,region,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,unit_cost,unit_price,promotion_flag,stockout_flag,demand_forecast
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,date,sku_id,warehouse_id,supplier_id,region,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,unit_cost,unit_price,promotion_flag,stockout_flag,demand_forecast
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


np.int64(0)

,Column,Method,Count,Rate,Status
0,units_sold,iqr,110,0.12%,warning
1,inventory_level,iqr,199,0.22%,warning
2,supplier_lead_time_days,zscore,0,0.00%,pass
3,reorder_point,iqr,0,0.00%,pass
4,order_quantity,iqr,5027,5.51%,warning
5,unit_cost,iqr,0,0.00%,pass
6,unit_price,iqr,0,0.00%,pass
7,demand_forecast,zscore,151,0.17%,warning


...

,status,current_dtype,expected_dtype
date,WARNING,str,datetime
sku_id,UKNOWN_COLUMN,str,NaN
warehouse_id,UKNOWN_COLUMN,str,NaN
supplier_id,UKNOWN_COLUMN,str,NaN
region,PASS,str,string
units_sold,PASS,int64,numeric
inventory_level,PASS,int64,numeric
supplier_lead_time_days,PASS,int64,numeric
reorder_point,PASS,int64,numeric
order_quantity,PASS,int64,numeric


Column Summary


{'numeric_columns': ['units_sold',
  'inventory_level',
  'supplier_lead_time_days',
  'reorder_point',
  'order_quantity',
  'unit_cost',
  'unit_price',
  'promotion_flag',
  'stockout_flag',
  'demand_forecast'],
 'categorical_columns': ['date',
  'sku_id',
  'warehouse_id',
  'supplier_id',
  'region'],
 'datetime_columns': [],
 'unique_counts': date                        365
 sku_id                       50
 warehouse_id                  5
 supplier_id                  10
 region                        4
 units_sold                   58
 inventory_level             781
 supplier_lead_time_days      13
 reorder_point               142
 order_quantity              301
 unit_cost                   230
 unit_price                  227
 promotion_flag                2
 stockout_flag                 1
 demand_forecast            4736
 dtype: int64}

Descriptive Statistics


{'data_description':             date sku_id warehouse_id supplier_id region    units_sold  \
 count      91250  91250        91250       91250  91250  91250.000000   
 unique       365     50            5          10      4           NaN   
 top     2024/1/1  SKU_1         WH_1       SUP_7  North           NaN   
 freq         250   1825        18250       12410  22974           NaN   
 mean         NaN    NaN          NaN         NaN    NaN     20.054564   
 std          NaN    NaN          NaN         NaN    NaN      9.068602   
 min          NaN    NaN          NaN         NaN    NaN      0.000000   
 25%          NaN    NaN          NaN         NaN    NaN     13.000000   
 50%          NaN    NaN          NaN         NaN    NaN     20.000000   
 75%          NaN    NaN          NaN         NaN    NaN     27.000000   
 max          NaN    NaN          NaN         NaN    NaN     59.000000   
 
         inventory_level  supplier_lead_time_days  reorder_point  \
 count      91250.0000

The profiling report indicates that the dataset contains missing values, outliers, duplicated records, and several business rule violations that require cleaning.

## Data cleaning

In [13]:
df["Date"].sample(20)

72233    2024/11/24
26338     2024/2/28
43488     2024/2/23
66149     2024/3/25
17654     2024/5/14
12845     2024/3/11
32651     2024/6/15
20373    2024/10/25
33026     2024/6/25
75461     2024/9/28
21284     2024/4/24
7259     2024/11/20
10221      2024/1/2
84799     2024/4/29
23745     2024/1/21
22651     2024/1/22
35781     2024/1/12
71613     2024/3/14
75417     2024/8/15
50093     2024/3/29
Name: Date, dtype: str

In [14]:
from src.cleaning import clean_data
cleaned_df, cleaning_report = clean_data(df, profile)
print("="*60)
print("missing_values")
display(cleaning_report["missing_values"])
print("="*60)
print("duplicate_rows")
display(cleaning_report["duplicate_rows"])
print("="*60)
print("column_names")
display(cleaning_report["column_names"])
print("="*60)
print("column_values")
display(cleaning_report["column_values"])
print("="*60)
print("data_types")
display(cleaning_report["data_types"])
print("="*60)
print("validate_business_rules")
display(cleaning_report["validate_business_rules"])
print("="*60)
print("outliers")
display(cleaning_report["outliers"])

Before convert:
0    2024/1/1
1    2024/1/2
2    2024/1/3
3    2024/1/4
4    2024/1/5
Name: date, dtype: str
Processing: units_sold
order_quantity
0      86223
386       29
426       28
413       28
218       28
Name: count, dtype: int64
Column: units_sold
Lower Bound: -8.0
Upper Bound: 48.0
Processing: inventory_level
order_quantity
0      86223
386       29
426       28
413       28
218       28
Name: count, dtype: int64
Processing: supplier_lead_time_days
order_quantity
0      86223
386       29
426       28
413       28
218       28
Name: count, dtype: int64
Processing: reorder_point
order_quantity
0      86223
386       29
426       28
413       28
218       28
Name: count, dtype: int64
Column: reorder_point
Lower Bound: 111.0
Upper Bound: 487.0
Processing: order_quantity
order_quantity
0      86223
386       29
426       28
413       28
218       28
Name: count, dtype: int64
Processing: unit_cost
order_quantity
0      86223
386       29
426       28
413       28
218       28
Name

{'filled_cells': 0, 'dropped_rows': 0, 'strategy': {}}

duplicate_rows


{'initial_row_count': 91250, 'final_row_count': 91250, 'removed_duplicates': 0}

column_names


...

{'date': {'from': 'str', 'to': 'str'},
 'sku_id': {'from': 'str', 'to': 'str'},
 'warehouse_id': {'from': 'str', 'to': 'str'},
 'supplier_id': {'from': 'str', 'to': 'str'},
 'region': {'from': 'str', 'to': 'str'},
 'units_sold': {'from': 'int64', 'to': 'int64'},
 'inventory_level': {'from': 'int64', 'to': 'int64'},
 'supplier_lead_time_days': {'from': 'int64', 'to': 'int64'},
 'reorder_point': {'from': 'int64', 'to': 'int64'},
 'order_quantity': {'from': 'int64', 'to': 'int64'},
 'unit_cost': {'from': 'float64', 'to': 'float64'},
 'unit_price': {'from': 'float64', 'to': 'float64'},
 'promotion_flag': {'from': 'int64', 'to': 'int64'},
 'stockout_flag': {'from': 'int64', 'to': 'int64'},
 'demand_forecast': {'from': 'float64', 'to': 'float64'}}

validate_business_rules


{'units_sold': {'rule': '>= 0',
  'severity': 'high',
  'action': 'drop',
  'invalid_count': 0,
  'invalid_index': [],
  'invalid_values': []},
 'inventory_level': {'rule': '>= 0',
  'severity': 'medium',
  'action': 'impute',
  'invalid_count': 0,
  'invalid_index': [],
  'invalid_values': []},
 'supplier_lead_time_days': {'rule': '> 0',
  'severity': 'high',
  'action': 'drop',
  'invalid_count': 0,
  'invalid_index': [],
  'invalid_values': []},
 'reorder_point': {'rule': '>= 0',
  'severity': 'medium',
  'action': 'impute',
  'invalid_count': 0,
  'invalid_index': [],
  'invalid_values': []},
 'order_quantity': {'rule': '>= 0',
  'severity': 'high',
  'action': 'drop',
  'invalid_count': 0,
  'invalid_index': [],
  'invalid_values': []},
 'unit_cost': {'rule': '> 0',
  'severity': 'high',
  'action': 'drop',
  'invalid_count': 0,
  'invalid_index': [],
  'invalid_values': []},
 'demand_forecast': {'rule': '>= 0',
  'severity': 'medium',
  'action': 'impute',
  'invalid_count': 0,
 

outliers


{'units_sold': {'method': 'iqr',
  'action': 'clip',
  'status': 'warning',
  'affected_rows': 110},
 'inventory_level': {'method': 'iqr',
  'action': 'ignore',
  'status': 'warning',
  'affected_rows': 199},
 'supplier_lead_time_days': {'method': 'zscore',
  'action': 'drop',
  'status': 'pass',
  'affected_rows': 0},
 'reorder_point': {'method': 'iqr',
  'action': 'clip',
  'status': 'pass',
  'affected_rows': 0},
 'order_quantity': {'method': 'iqr',
  'action': 'ignore',
  'status': 'warning',
  'affected_rows': 5027},
 'unit_cost': {'method': 'iqr',
  'action': 'clip',
  'status': 'pass',
  'affected_rows': 0},
 'unit_price': {'method': 'iqr',
  'action': 'clip',
  'status': 'pass',
  'affected_rows': 0},
 'demand_forecast': {'method': 'zscore',
  'action': 'flag',
  'status': 'warning',
  'affected_rows': 151}}

In [15]:
cleaned_df["order_quantity"].value_counts().head(20)

order_quantity
0      86223
386       29
426       28
413       28
218       28
440       27
429       27
453       26
257       26
209       26
225       25
201       25
302       25
488       24
301       24
255       24
382       24
478       24
470       23
452       23
Name: count, dtype: int64

## Feature Engineering

In [16]:
from src.feature_engineer import feature_engineer
feature_df, feature_report = feature_engineer(cleaned_df)

display(feature_report)
display(feature_df.head())

{'financial': {'created_features': ['revenue',
   'cogs',
   'profit',
   'profit_margin'],
  'formulas': {'revenue': 'units_sold * unit_price',
   'cogs': 'units_sold * unit_cost',
   'profit': 'revenue - cogs',
   'profit_margin': 'profit / revenue'}},
 'inventory': {'created_features': ['inventory_value',
   'inventory_gap',
   'reorder_risk_flag',
   'days_of_supply'],
  'formulas': {'inventory_value': 'inventory_level * unit_cost',
   'inventory_gap': 'inventory_level - reorder_point',
   'reorder_risk_flag': 'inventory_level < reorder_point',
   'days_of_supply': 'inventory_level / demand_forecast'},
  'assumptions': {'days_of_supply': 'demand_forecast represents daily demand.'}},
 'time': {'created_features': ['year',
   'month',
   'month_name',
   'day',
   'quarter']},
 'supplier': {'created_features': ['lead_time_category',
   'lead_time_demand',
   'projected_inventory_after_lead_time',
   'lead_time_reorder_risk',
   'projected_stockout_risk'],
  'formulas': {'lead_time_de

,date,sku_id,warehouse_id,supplier_id,region,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,...,day,quarter,lead_time_category,lead_time_demand,projected_inventory_after_lead_time,lead_time_reorder_risk,projected_stockout_risk,forecast_error,absolute_forecast_error,forecast_direction
0,2024-01-01,sku_1,wh_1,sup_8,west,10,592,14,379,0,...,1,1,slow,119.28,472.72,False,False,-1.48,1.48,underforecast
1,2024-01-02,sku_1,wh_1,sup_8,west,17,575,14,379,0,...,2,1,slow,260.82,314.18,True,False,1.63,1.63,overforecast
2,2024-01-03,sku_1,wh_1,sup_8,north,35,540,14,379,0,...,3,1,slow,554.68,-14.68,True,True,4.62,4.62,overforecast
3,2024-01-04,sku_1,wh_1,sup_8,south,24,516,14,379,0,...,4,1,slow,272.02,243.98,True,False,-4.57,4.57,underforecast
4,2024-01-05,sku_1,wh_1,sup_8,west,21,495,14,379,0,...,5,1,slow,261.80,233.20,True,False,-2.30,2.30,underforecast


In [17]:
feature_df["order_quantity"].sample(20)

12016      0
30452      0
33439      0
9205       0
90306      0
66462      0
5121       0
30809      0
15295      0
55381      0
35212      0
43630      0
75338      0
29169      0
77140      0
32567      0
57846      0
55548      0
45006    321
628        0
Name: order_quantity, dtype: int64

## Final Dataset Summary

In [18]:
display(feature_df.describe())

,date,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,unit_cost,unit_price,promotion_flag,stockout_flag,...,inventory_value,inventory_gap,year,month,day,quarter,lead_time_demand,projected_inventory_after_lead_time,forecast_error,absolute_forecast_error
count,91250,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.0,...,91250.000000,91250.000000,91250.0,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000
mean,2024-07-01 00:00:00,20.051156,471.522312,7.984000,300.068000,19.272493,12.203320,18.261800,0.101589,0.0,...,5756.098533,171.454312,2024.0,6.498630,15.715068,2.501370,160.359884,311.162428,0.030877,2.381533
min,2024-01-01 00:00:00,0.000000,168.000000,2.000000,201.000000,0.000000,5.020000,6.950000,0.000000,0.0,...,1094.160000,-49.000000,2024.0,1.000000,1.000000,1.000000,0.000000,-541.660000,-13.000000,0.000000
25%,2024-04-01 00:00:00,13.000000,370.000000,4.000000,252.000000,0.000000,8.180000,12.000000,0.000000,0.0,...,3521.070000,71.000000,2024.0,4.000000,8.000000,2.000000,67.800000,193.617500,-1.990000,0.950000
50%,2024-07-01 00:00:00,20.000000,461.000000,8.000000,300.000000,0.000000,11.990000,18.180000,0.000000,0.0,...,5247.550000,160.000000,2024.0,7.000000,16.000000,3.000000,133.420000,313.600000,0.010000,2.000000
75%,2024-09-30 00:00:00,27.000000,564.000000,11.000000,346.000000,0.000000,16.320000,23.390000,0.000000,0.0,...,7526.787500,258.000000,2024.0,9.000000,23.000000,3.000000,230.355000,434.677500,2.030000,3.440000
max,2024-12-30 00:00:00,48.000000,990.000000,14.000000,398.000000,499.000000,19.760000,35.100000,1.000000,0.0,...,19186.960000,744.000000,2024.0,12.000000,31.000000,4.000000,835.380000,957.840000,14.080000,14.080000
std,NaN,9.057199,133.488002,3.907929,54.879945,82.340831,4.574982,7.121136,0.302109,0.0,...,2788.542737,122.626009,0.0,3.443993,8.787394,1.116813,115.436038,179.120087,2.990064,1.808225


In [19]:
feature_df.info()
feature_df.shape
feature_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 91250 entries, 0 to 91249
Data columns (total 37 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   date                                 91250 non-null  datetime64[us]
 1   sku_id                               91250 non-null  str           
 2   warehouse_id                         91250 non-null  str           
 3   supplier_id                          91250 non-null  str           
 4   region                               91250 non-null  str           
 5   units_sold                           91250 non-null  int64         
 6   inventory_level                      91250 non-null  int64         
 7   supplier_lead_time_days              91250 non-null  int64         
 8   reorder_point                        91250 non-null  int64         
 9   order_quantity                       91250 non-null  int64         
 10  unit_cost            

,date,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,unit_cost,unit_price,promotion_flag,stockout_flag,...,inventory_value,inventory_gap,year,month,day,quarter,lead_time_demand,projected_inventory_after_lead_time,forecast_error,absolute_forecast_error
count,91250,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.0,...,91250.000000,91250.000000,91250.0,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000
mean,2024-07-01 00:00:00,20.051156,471.522312,7.984000,300.068000,19.272493,12.203320,18.261800,0.101589,0.0,...,5756.098533,171.454312,2024.0,6.498630,15.715068,2.501370,160.359884,311.162428,0.030877,2.381533
min,2024-01-01 00:00:00,0.000000,168.000000,2.000000,201.000000,0.000000,5.020000,6.950000,0.000000,0.0,...,1094.160000,-49.000000,2024.0,1.000000,1.000000,1.000000,0.000000,-541.660000,-13.000000,0.000000
25%,2024-04-01 00:00:00,13.000000,370.000000,4.000000,252.000000,0.000000,8.180000,12.000000,0.000000,0.0,...,3521.070000,71.000000,2024.0,4.000000,8.000000,2.000000,67.800000,193.617500,-1.990000,0.950000
50%,2024-07-01 00:00:00,20.000000,461.000000,8.000000,300.000000,0.000000,11.990000,18.180000,0.000000,0.0,...,5247.550000,160.000000,2024.0,7.000000,16.000000,3.000000,133.420000,313.600000,0.010000,2.000000
75%,2024-09-30 00:00:00,27.000000,564.000000,11.000000,346.000000,0.000000,16.320000,23.390000,0.000000,0.0,...,7526.787500,258.000000,2024.0,9.000000,23.000000,3.000000,230.355000,434.677500,2.030000,3.440000
max,2024-12-30 00:00:00,48.000000,990.000000,14.000000,398.000000,499.000000,19.760000,35.100000,1.000000,0.0,...,19186.960000,744.000000,2024.0,12.000000,31.000000,4.000000,835.380000,957.840000,14.080000,14.080000
std,NaN,9.057199,133.488002,3.907929,54.879945,82.340831,4.574982,7.121136,0.302109,0.0,...,2788.542737,122.626009,0.0,3.443993,8.787394,1.116813,115.436038,179.120087,2.990064,1.808225


In [20]:
display(feature_df.columns.to_list())

['date',
 'sku_id',
 'warehouse_id',
 'supplier_id',
 'region',
 'units_sold',
 'inventory_level',
 'supplier_lead_time_days',
 'reorder_point',
 'order_quantity',
 'unit_cost',
 'unit_price',
 'promotion_flag',
 'stockout_flag',
 'demand_forecast',
 'demand_forecast_is_outlier',
 'revenue',
 'cogs',
 'profit',
 'profit_margin',
 'inventory_value',
 'inventory_gap',
 'reorder_risk_flag',
 'days_of_supply',
 'year',
 'month',
 'month_name',
 'day',
 'quarter',
 'lead_time_category',
 'lead_time_demand',
 'projected_inventory_after_lead_time',
 'lead_time_reorder_risk',
 'projected_stockout_risk',
 'forecast_error',
 'absolute_forecast_error',
 'forecast_direction']

In [21]:
feature_df.isnull().sum()

date                                     0
sku_id                                   0
warehouse_id                             0
supplier_id                              0
region                                   0
units_sold                               0
inventory_level                          0
supplier_lead_time_days                  0
reorder_point                            0
order_quantity                           0
unit_cost                                0
unit_price                               0
promotion_flag                           0
stockout_flag                            0
demand_forecast                          0
demand_forecast_is_outlier               0
revenue                                  0
cogs                                     0
profit                                   0
profit_margin                          620
inventory_value                          0
inventory_gap                            0
reorder_risk_flag                        0
days_of_sup

## Export Dataset

In [22]:
feature_df.to_csv(
    "../data/processed/inventory_clean.csv",
    index=False
)